# Fraud detection: Gemma 3 1B + GAT

In [ ]:
# Install once in the selected notebook environment if needed:
# %pip install -r requirements.txt
# %pip install bitsandbytes>=0.43

from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import GATConv
from torch_geometric.loader import NeighborLoader
from sklearn.metrics import f1_score, roc_auc_score
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model

SEED = 42
DATASET_PATH = Path("datasets/reddit.pt")  # change to datasets/instagram.pt when needed
OUTPUT_DIR = Path("artifacts/notebook-reddit")
MODEL_NAME = "google/gemma-3-1b-it"
HIDDEN_DIM, HEADS = 128, 4
NEIGHBORS_PER_NODE = 10
MAX_INPUT_TOKENS, MAX_NEW_TOKENS = 2048, 64
EPOCHS, GAT_LR, LLM_LR = 3, 1e-3, 1e-4
ALPHA, BETA = 0.1, 0.1
BATCH_SIZE, VAL_BATCH_SIZE = 8, 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(DEVICE, DATASET_PATH)


/home/jyo/Desktop/Projects/LLM_Enhanced_GNN_Fraud_Detection/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/jyo/Desktop/Projects/LLM_Enhanced_GNN_Fraud_Detection/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device=cuda, dataset=datasets/reddit.pt


### Data loading and split handling

In [ ]:

try:
    graph = torch.load(DATASET_PATH, map_location="cpu", weights_only=False)
except TypeError:
    graph = torch.load(DATASET_PATH, map_location="cpu")
if not hasattr(graph, "raw_texts") or not hasattr(graph, "edge_index") or not hasattr(graph, "y"):
    raise ValueError("Dataset must contain raw_texts, edge_index, and y")
graph.raw_texts = [str(text) for text in graph.raw_texts]
graph.edge_index = graph.edge_index.long().cpu()
graph.y = graph.y.long().view(-1).cpu()
def split_indices(name):
    mask = getattr(graph, name + "_mask", None)
    if mask is None:
        raise ValueError(f"Dataset has no {name}_mask")
    return mask.nonzero(as_tuple=False).view(-1).long()
train_idx, val_idx = split_indices("train"), split_indices("val")
print(graph, "train nodes:", len(train_idx), "validation nodes:", len(val_idx))

In [ ]:
class GemmaEnhancer:
    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        base = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=DTYPE,
        )
        self.model = get_peft_model(
            base,
            LoraConfig(
                task_type=TaskType.CAUSAL_LM,
                r=8,
                lora_alpha=16,
                lora_dropout=0.05,
                target_modules=["q_proj", "v_proj"],
                bias="none",
            ),
        )
        self.model.gradient_checkpointing_enable()

    def prompt(self, text, causal):
        focus = "discriminative features related to the fraud label" if causal else "generic background information unrelated to the fraud label"
        return f"Extract one concise sentence containing {focus}. Do not predict the label. Text: {text[:4000]}\nAnswer:"

    def _device(self):
        return next(self.model.parameters()).device

    @torch.no_grad()
    def generate(self, texts):
        prompts = [self.prompt(text, causal) for text in texts for causal in (True, False)]
        inputs = self.tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_INPUT_TOKENS)
        inputs = {k: v.to(self._device()) for k, v in inputs.items()}
        tokens = self.model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=self.tokenizer.pad_token_id,
        )
        output_ids = tokens[:, inputs["input_ids"].shape[1]:]
        decoded = self.tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        decoded = [item.strip().splitlines()[0] if item.strip().splitlines() else "" for item in decoded]
        pairs = []
        for idx in range(0, len(decoded), 2):
            pairs.append((decoded[idx], decoded[idx + 1]))
        return pairs

    def encode(self, texts):
        inputs = self.tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_INPUT_TOKENS)
        inputs = {k: v.to(self._device()) for k, v in inputs.items()}
        hidden = self.model.base_model.model(**inputs, output_hidden_states=True, return_dict=True).last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1)
        return (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)


enhancer = GemmaEnhancer()
enhancer.model.print_trainable_parameters()


In [ ]:
class GATClassifier(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.first = GATConv(in_dim, HIDDEN_DIM, heads=HEADS, concat=True, dropout=0.5)
        self.second = GATConv(HIDDEN_DIM * HEADS, 2, heads=1, concat=False, dropout=0.5)
    def forward(self, x, edge_index):
        x = F.dropout(x, 0.5, self.training)
        x = self.first(x, edge_index).elu()
        return self.second(F.dropout(x, 0.5, self.training), edge_index)
def g_loss(causal_logits, residual_logits, labels):
    disc = F.cross_entropy(causal_logits, labels)
    uniform = torch.full_like(residual_logits, 0.5)
    residual = F.kl_div(F.log_softmax(residual_logits, -1), uniform, reduction="batchmean")
    orthogonal = F.cosine_similarity(causal_logits.flatten(), residual_logits.flatten(), dim=0)
    return disc + ALPHA * residual + BETA * orthogonal
def induced_edges(indices):
    nodes = indices.cpu(); mapping = torch.full((graph.num_nodes,), -1, dtype=torch.long); mapping[nodes] = torch.arange(len(nodes))
    edges = graph.edge_index; keep = (mapping[edges[0]] >= 0) & (mapping[edges[1]] >= 0)
    return mapping[edges[:, keep]].to(DEVICE)
@torch.no_grad()
def semantic_filter(edges, embeddings):
    normalized = F.normalize(embeddings.float(), dim=-1)
    scores = (normalized[edges[0]] * normalized[edges[1]]).sum(-1)
    keep = torch.zeros(edges.shape[1], dtype=torch.bool, device=edges.device)
    for source in edges[0].unique():
        candidates = (edges[0] == source).nonzero().view(-1)
        chosen = torch.topk(scores[candidates], min(NEIGHBORS_PER_NODE, len(candidates))).indices
        keep[candidates[chosen]] = True
    return edges[:, keep]

# Independent alternating training and validation




In [ ]:
def make_loader(indices, batch_size, shuffle=False):
    if len(indices) == 0:
        return None
    return NeighborLoader(
        data=graph,
        num_neighbors=[-1] * 2,
        batch_size=batch_size,
        input_nodes=indices,
        shuffle=shuffle,
        directed=False,
    )


def summarize_metrics(loss, labels, logits):
    prediction = logits.argmax(-1).cpu().numpy()
    probability = logits.softmax(-1)[:, 1].cpu().numpy()
    metrics = {"loss": float(loss.detach().cpu())}
    metrics["f1"] = f1_score(labels.cpu(), prediction, average="macro")
    metrics["auc"] = roc_auc_score(labels.cpu(), probability) if len(set(labels.cpu().tolist())) > 1 else float("nan")
    return metrics


def run_split(indices, training, batch_size=BATCH_SIZE):
    loader = make_loader(indices, batch_size=batch_size, shuffle=training)
    if loader is None:
        return {"loss": 0.0, "f1": 0.0, "auc": float("nan")}

    totals = {"loss": 0.0, "count": 0}
    all_labels = []
    all_preds = []
    all_probs = []

    for batch in loader:
        batch_idx = batch.n_id.to(torch.long)
        batch_texts = [graph.raw_texts[int(i)] for i in batch_idx.tolist()]
        pairs = enhancer.generate(batch_texts)
        causal = enhancer.encode([pair[0] if len(pair) > 0 else text[:200] for pair, text in zip(pairs, batch_texts)])
        residual = enhancer.encode([pair[1] if len(pair) > 1 else text[:200] for pair, text in zip(pairs, batch_texts)])
        with torch.no_grad():
            original = enhancer.encode(batch_texts)
        edges = semantic_filter(induced_edges(batch_idx.cpu()), original)
        labels = graph.y[batch_idx].to(DEVICE)

        gat.train(training)
        enhancer.model.train(training)
        if training:
            llm_optimizer.zero_grad(); gat_optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            causal_logits = gat(causal, edges)
            residual_logits = gat(residual, edges)
            loss = g_loss(causal_logits, residual_logits, labels)
            if training:
                loss.backward()
                llm_optimizer.step()
                gat_optimizer.step()

        if not training:
            with torch.no_grad():
                logits = gat(causal, edges)
                total_loss = g_loss(logits, gat(residual, edges), labels)
                totals["loss"] += total_loss.item() * len(batch_idx)
                totals["count"] += len(batch_idx)
                prediction = logits.argmax(-1).cpu().numpy()
                probability = logits.softmax(-1)[:, 1].cpu().numpy()
                all_labels.append(labels.cpu())
                all_preds.append(prediction)
                all_probs.append(probability)

    if training:
        return {"loss": float(loss.detach().cpu()), "f1": 0.0, "auc": float("nan")}

    labels_all = torch.cat(all_labels, dim=0)
    preds_all = np.concatenate(all_preds, axis=0)
    probs_all = np.concatenate(all_probs, axis=0)
    metrics = {"loss": totals["loss"] / max(totals["count"], 1)}
    metrics["f1"] = f1_score(labels_all, preds_all, average="macro")
    metrics["auc"] = roc_auc_score(labels_all, probs_all) if len(set(labels_all.tolist())) > 1 else float("nan")
    return metrics


with torch.no_grad():
    embedding_dim = enhancer.encode([graph.raw_texts[0]]).shape[-1]


gat = GATClassifier(embedding_dim).to(DEVICE)
llm_optimizer = torch.optim.AdamW(enhancer.model.parameters(), lr=LLM_LR)
gat_optimizer = torch.optim.AdamW(gat.parameters(), lr=GAT_LR, weight_decay=5e-4)

best = -float("inf")
training_history = []

for epoch in range(EPOCHS):
    train_metrics = run_split(train_idx, True, batch_size=BATCH_SIZE)
    val_metrics = run_split(val_idx, False, batch_size=VAL_BATCH_SIZE)
    score = val_metrics["f1"] + val_metrics["auc"]
    epoch_record = {
        "epoch": epoch + 1,
        "train_loss": train_metrics["loss"],
        "val_loss": val_metrics["loss"],
        "train_f1": train_metrics["f1"],
        "val_f1": val_metrics["f1"],
        "train_auc": train_metrics["auc"],
        "val_auc": val_metrics["auc"],
        "score": score,
    }
    training_history.append(epoch_record)
    print(f"epoch {epoch + 1}: train={train_metrics} val={val_metrics}")

    if score > best:
        best = score
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        torch.save(gat.state_dict(), OUTPUT_DIR / "gat.pt")
        enhancer.model.save_pretrained(OUTPUT_DIR / "gemma-lora")

history_df = pd.DataFrame(training_history)
print(history_df)

plt.figure(figsize=(12, 5))
sns.heatmap(history_df[["train_loss", "val_loss", "train_f1", "val_f1", "train_auc", "val_auc"]].T, annot=True, fmt=".3f", cmap="viridis")
plt.title("Training metrics by epoch")
plt.ylabel("Metric")
plt.xlabel("Epoch")
plt.xticks(np.arange(len(history_df)) + 0.5, [f"E{e['epoch']}" for e in history_df])
plt.tight_layout()
plt.show()

print("saved to", OUTPUT_DIR)
